# Data and downlink budget — how it works

This is the explanation and reference for the `quicksat` data budget: what the input files are, how a payload's data generation is turned into a volume, how the downlink is compared against it, and where the tool stops.

It is deliberately not a tutorial. In [Diátaxis](https://diataxis.fr/) terms, `sample/` holds the tutorials and how-to guides; `docs/` holds the explanation and the reference. It still runs, against the same sample data, because an explanation that cannot be executed drifts from the code it describes.

Every figure below comes from calling `quicksat.dataflow.budget` rather than from reproducing its arithmetic here, so the explanation and the module cannot quietly disagree about what the chain computes.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_
from quicksat.dataflow.budget import DataBudget, DataFlowModel
from quicksat.utils.orbit import Orbit

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [2]:
orbit = Orbit.from_yaml_file(Path("sample/data/orbit.yaml"))
model = DataFlowModel.from_yaml_file(Path("sample/data/pl_dataflow_model.yaml"))
budget = DataBudget(model, orbit)

generation, downlink = model.generation, model.downlink
budget

## The data model

A data budget (unlike the mass budget which sums muli) is **one chain of calculations** running from a handful of assumptions to a single comparison — does the downlink clear what the payload generates? Its inputs are not an inventory of things that exist; they are estimates of how the mission will behave. A duty cycle, a compression ratio and an average pass duration are all guesses, and none of them is a line item.

So the input is config rather than CSV, and the output is a chain rather than an aggregation:

```
period          = 2*pi*sqrt(r^3/mu)              r = earth_radius + altitude
orbits_per_day  = 1 day / period

gen_rate        = raw_datarate / compression_ratio
gen_duration    = duty_cycle * period
generated/orbit = gen_rate * gen_duration
generated/day   = generated/orbit * orbits_per_day

contact/day     = contacts_per_day * avg_contact_duration
downlink/day    = downlink_rate * contact/day
downlink/orbit  = downlink/day / orbits_per_day

margin          = downlink/day / generated/day - 1
storage         = generated/orbit * orbits_without_contact
```

Every line below is one of those, and each one is a property on `DataBudget`, computed on access. There is no `resolve()` to call first and nothing cached to invalidate, which is the other half of the difference from the mass budget: with no rows to filter there is no working table to build, only a handful of scalars that fall out of the inputs.

## The files

Two, and the split is deliberate.

**`sample/data/orbit.yaml`** is the shared orbit, loaded into `quicksat.utils.orbit.Orbit`. This budget takes two things from it — the orbital period and how many orbits fit in a day — and needs nothing else; the inclination it ignores entirely.

How the orbit is specified, validated and derived is in **`orbit_ref.ipynb`**. It lives there rather than here because the delta-V budget reads the same file for its own purposes, and agility will, so the explanation belongs with the thing rather than with one of its consumers.

**`sample/data/pl_dataflow_model.yaml`** is the payload dataflow model — generation, downlink and storage together. These stay in one file precisely because the budget exists to compare the first two: splitting the two halves of a comparison across files would make it harder to read, not easier.

| section | setting | meaning |
|---|---|---|
| `generation` | `raw_datarate` | instrument output *before* compression |
| | `compression_ratio` | divides the raw rate; 1 means no compression |
| | `duty_cycle` | fraction of each orbit spent generating |
| `downlink` | `rate` | achieved throughput |
| | `contacts_per_day` | usable ground contacts in a day, from Mission Analysis |
| | `avg_contact_duration` | how long an average contact lasts |
| `storage` | `orbits_without_contact` | the sizing case for onboard memory |

Contact statistics are **inputs**, not derived. How many contacts a day the mission gets, and how long they last, are numbers Mission Analysis hands over.

In [3]:
print(Path("sample/data/pl_dataflow_model.yaml").read_text())

# Payload dataflow model: what the instrument generates, what the downlink clears,
# and what has to be held on board in between.
#
# Orbit comes from orbit.yaml.

generation:
  raw_datarate: 1000 Mbit/s   # before compression; 416.67 Mbit/s after
  compression_ratio: 2.4
  duty_cycle: 5 %          # of each orbit: daytime, full VNIR

downlink:
  rate: 800 Mbit/s              # X-band transmitter throughput
  contacts_per_day: 7           # usable ground contacts, from Mission Analysis
  avg_contact_duration: 6 min   # average usable contact

storage:
  orbits_without_contact: 3   # sizing case for the mass memory



## Units, and one trap worth knowing

Every rate and duration is entered as text with its unit and parsed with [pint](https://pint.readthedocs.io/), the same as `unit_mass` in the equipment file. That buys the dimension check: a length where a data rate belongs is not a number that happens to be wrong, it is not a rate at all.

The trap is that **pint reads `bps` as baud**. Baud is symbols per second, and for a data rate it is numerically identical but semantically the wrong quantity — a symbol can carry more than one bit. Writing `800 Mbps` parses and computes correctly, then renders as `800 MBd` in any output, asserting a symbol rate that the sample file's own comment explicitly denies.

The sample files therefore write `Mbit/s`. It costs four characters and removes the ambiguity.

In [4]:
for text in ("800 Mbps", "800 Mbit/s"):
    q = Q_(text)
    print(f"{text:12s} -> {q:~}   (dimensionality {q.dimensionality})")

print("\nnumerically identical:", Q_("800 Mbps") == Q_("800 Mbit/s"))
print("but 'Bd' is symbols/second, not bits/second - the wrong quantity to assert")

# and the dimension check cannot help here: bits are dimensionless in pint, so a
# data rate and a frequency are the same dimension
print(f"\n800 MHz is also {Q_('800 MHz').dimensionality}, so DataRateQty accepts it")

# a duty cycle is a dimensionless percentage, and pint keeps that straight
print(f"\nduty cycle {generation.duty_cycle:~} -> "
      f"{generation.duty_cycle.to('dimensionless'):~.3f} as a fraction")

800 Mbps     -> 800 MBd   (dimensionality 1 / [time])
800 Mbit/s   -> 800.0 Mbit / s   (dimensionality 1 / [time])

numerically identical: True
but 'Bd' is symbols/second, not bits/second - the wrong quantity to assert

800 MHz is also 1 / [time], so DataRateQty accepts it

duty cycle 5 % -> 0.050 as a fraction


## The orbit

Every per-day figure in this budget is a per-orbit figure multiplied by how many orbits fit in a day, so those two quantities are all the budget takes from the orbit:

$$T = 2\pi\sqrt{r^3/\mu}, \quad r = R_\oplus + h \qquad n_{\text{day}} = \frac{1\,\text{day}}{T}$$

`orbits_per_day` is dimensionless rather than a rate, so it can multiply a per-orbit volume without pint objecting that orbits are not seconds. The constants behind μ, what is checked on load, and what the circular assumption costs are all in `orbit_ref.ipynb`.

In [5]:
period = orbit.period
orbits_per_day = orbit.orbits_per_day

print(f"altitude        {orbit.altitude:~.1f}")
print(f"period          {period:~.0f}  ({period.to('min'):~.2f})")
print(f"orbits per day  {orbits_per_day:~.3f}")

altitude        500.0 km
period          5677 s  (94.62 min)
orbits per day  15.219


## Generation

The payload generates at some rate for some fraction of each orbit.

**Compression is modelled rather than folded in.** The file states the raw instrument data rate and the compression ratio separately, so the ratio can be varied. Quoting only the post-compression rate would hide the one input most likely to be argued about.

**The duty cycle is a fraction of the orbit, not of the daylight pass.** 5% of a 5677 s orbit is 284 s of imaging.

In [6]:
print(f"raw rate          {generation.raw_datarate:~.2f}")
print(f"compression       {generation.compression_ratio}x")
print(f"effective rate    {budget.effective_datarate:~.2f}")
print(f"generation window {budget.generation_duration:~.2f} per orbit")
print()
print(f"generated         {budget.generated_per_orbit:~.2f} per orbit")
print(f"                  {budget.generated_per_day:~.2f} per day")

raw rate          1000.00 Mbit / s
compression       2.4x
effective rate    416.67 Mbit / s
generation window 283.85 s per orbit

generated         14.78 GB per orbit
                  225.00 GB per day


## Downlink

The same shape as generation, from the other side: a rate multiplied by a duration. What differs is the direction the chain runs.

Generation is naturally **per orbit** — the payload collects for some fraction of each revolution — and the daily figure is derived from it. Downlink is naturally **per day**: contacts are counted against the ground station's day, not the spacecraft's orbit, and the count does not divide evenly into orbits anyway. So the two halves of the comparison are computed in their own natural periods and meet at the daily figure.

`contacts_per_day` and `avg_contact_duration` multiply to the daily contact time. Quoting them separately rather than as one number is deliberate: it is the form a ground segment team uses, and it keeps visible which of the two assumptions is doing the work when the budget moves.

In [7]:
print(f"downlink rate     {downlink.rate:~.0f}")
print(f"contacts          {downlink.contacts_per_day:g} per day at "
      f"{downlink.avg_contact_duration:~.0f} each")
print(f"contact time      {budget.contact_per_day:~.1f} per day")
print()
print(f"downlinked        {budget.downlinked_per_day:~.2f} per day")
print(f"                  {budget.downlinked_per_orbit:~.2f} per orbit "
      f"(derived, {orbits_per_day:~.3f} orbits/day)")

downlink rate     800 Mbit / s
contacts          7 per day at 6 min each
contact time      42.0 min per day

downlinked        252.00 GB per day
                  16.56 GB per orbit (derived, 15.219 orbits/day)


## The margin, which is the point

Everything above exists to produce the margin.

$$\text{margin} = \frac{\text{downlinked}}{\text{generated}} - 1$$

A positive margin means the link clears the backlog; negative means data accumulates until something is deleted or a pass is added. Being a ratio, it is the one figure in this budget that **the byte convention cannot affect** — it survives any consistent choice of GB or GiB, because the convention cancels.

What that margin then implies for a particular mission — how far each assumption can move before it is gone, and which input to reach for when it is — is a question about that mission rather than about the model. `sample/data_budget.ipynb` works it through for the sample satellite.

In [8]:
margin = budget.margin
balance = budget.downlinked_per_day - budget.generated_per_day

print(f"generated    {budget.generated_per_day:~.2f} per day")
print(f"downlinked   {budget.downlinked_per_day:~.2f} per day")
print(f"margin       {margin.to('percent'):~+.2f}")
print(f"{'surplus' if balance > 0 else 'shortfall':12s} {abs(balance):~.2f} per day")

generated    225.00 GB per day
downlinked   252.00 GB per day
margin       +12.00 %
surplus      27.00 GB per day


## Bytes: decimal, not binary

quicksat uses **decimal** bytes throughout — `GB` is 10⁹ bytes. Link rates are quoted decimal, so this keeps one convention across the whole chain, and pint enforces it rather than leaving it to a comment.

It is worth stating plainly because a volume in bytes is ambiguous unless the convention is named, and the common habit of dividing megabits by 8 and then by 1024 mixes the two — decimal megabits into binary gigabytes. All three readings describe the same physical quantity and differ by up to about 2.3%, as the table below shows. Someone reconciling two figures by hand will find that gap and reasonably suspect a bug; it is a convention, not an error.

Everything convention-independent is unaffected: contact minutes, contact count, and the margin.

In [9]:
raw_bits = budget.effective_datarate * budget.generation_duration
comparison = pd.DataFrame(
    {
        "per orbit": [
            raw_bits.to("GB").magnitude,
            raw_bits.to("GiB").magnitude,
            raw_bits.to("Mbit").magnitude / 8 / 1024,
        ]
    },
    index=["decimal GB (quicksat)", "binary GiB", "mixed (Mbit / 8 / 1024)"],
)
comparison["vs quicksat"] = (
    comparison["per orbit"] / comparison.loc["decimal GB (quicksat)", "per orbit"] - 1
) * 100
comparison

,per orbit,vs quicksat
decimal GB (quicksat),14.784,0.000
binary GiB,13.768,-6.868
mixed (Mbit / 8 / 1024),14.437,-2.344


## Storage

The onboard memory has to hold whatever accumulates between contacts. quicksat sizes this the simple way — the data generated across a stated number of orbits without a pass — rather than by simulating an accumulating backlog over the mission.

That is a deliberately crude model, and it is the right crudeness for a sizing tool: the input is "how many orbits might we go without a usable pass", which a ground segment team can answer, rather than a contact schedule nobody has yet.

Note what this figure does **not** capture. It sizes the gap between passes, not an accumulating backlog — and the distinction matters, because the two coincide only while the budget closes. Were the margin negative the backlog would never clear, storage demand would grow without bound, and this number would be meaningless rather than merely approximate.

In [10]:
print(f"generated per orbit      {budget.generated_per_orbit:~.2f}")
print(f"orbits without contact   {model.storage.orbits_without_contact:g}")
print(f"storage required         {budget.storage_required():~.2f}")
print(f"  overridable:           {budget.storage_required(1):~.2f} for a single orbit")

# what the mass budget says is actually flying
equipment = pd.read_csv("sample/data/equipment.csv")
ssdr = equipment.query("equipment_id == 'mass_memory'").iloc[0]
print(f"\nflying, per the equipment list: {ssdr['equipment_name']} ({ssdr['unit_mass']})")
print(f"  required {budget.storage_required().to('TB'):~.3f} against a 2 TB device - ample,")
print("  before any allowance for file system overhead, redundancy or degradation")

generated per orbit      14.78 GB
orbits without contact   3
storage required         44.35 GB
  overridable:           14.78 GB for a single orbit



flying, per the equipment list: SSDR 2 TB (4.5 kg)
  required 0.044 TB against a 2 TB device - ample,
  before any allowance for file system overhead, redundancy or degradation


## The class

`DataFlowModel` and `Orbit` are the two inputs, and `DataBudget` is the chain over them. Both models validate on load, so a malformed file names the offending field rather than failing later in the arithmetic.

There are two ways in, and which one to reach for depends on whether anything else needs the orbit:

- **`DataBudget.from_yaml_file(model_path, orbit_path)`** reads both files. One budget, one line, nothing else to keep in step.
- **`DataBudget(model, orbit)`** takes objects already loaded. This is the one to use when the delta-V budget is in the same notebook: the orbit is then parsed once and every budget is demonstrably flying the same one, rather than three files that merely agree today.

Everything after that is a property, named for what it is *per*: `effective_datarate`, `generation_duration`, `generated_per_orbit`, `generated_per_day`, `contact_per_day`, `downlinked_per_day`, `downlinked_per_orbit`, and `margin`. `storage_required()` is the only method, and only because it takes an override.

There is no setter anywhere. To ask a what-if, build another `DataFlowModel` — which is one line from a dict, and keeps the case you asked about distinguishable from the case in the file.

### The document view

`tabulated_data()` lays the whole chain out as a document: item, value, unit, and the assumption behind it, in the order the chain computes them. What comes back is a pandas `Styler`, so the figures print at a fixed number of decimals and the two rows carrying an answer rather than a step — the margin and the storage — come out bold.

The frame is still there as `.data`, and it keeps the one column the render hides. Every row is tagged `input` for what was typed into a file, `derived` for what the chain worked out from it, then `margin` and `storage` for the two results. That tag is what makes the report filterable: the inputs of one case can be pulled out and diffed against another without parsing captions.

In [11]:
budget.tabulated_data()

,Value,Unit,Comment
Orbital period,94.62,min,500 km circular
Orbits per day,15.22,-,
Data generation rate,416.67,Mbit/s,"1000 Mbit / s raw, 2.4x compression"
Data generation duration,283.85,s/orbit,5 % duty cycle
Data generated,14.78,GB/orbit,
,225.00,GB/day,
Data downlink rate,800.00,Mbit/s,achieved throughput
Contacts,7.00,per day,6 min average duration
Data downlink duration,42.00,min/day,
Data downlinked,16.56,GB/orbit,


In [12]:
report = budget.tabulated_data().data
print(report["row_type"].value_counts().to_dict())

# what was typed into a file, as opposed to what the chain worked out
print()
print(report.query("row_type == 'input'")[["item", "value", "unit"]].to_string(index=False))

{'derived': 6, 'input': 5, 'margin': 1, 'storage': 1}

                    item   value    unit
          Orbital period  94.616     min
    Data generation rate 416.667  Mbit/s
Data generation duration 283.849 s/orbit
      Data downlink rate 800.000  Mbit/s
                Contacts   7.000 per day


## Limitations

What the data budget deliberately does not do, in rough order of how often it comes up:

- **No contact schedule.** Passes are an average count and an average duration. There is no station list, no elevation mask, no propagator, and therefore no notion of a gap at a particular time of day — which is exactly the thing that sizes storage in a real design.
- **Storage is a multiple of one orbit's generation.** No accumulating backlog, no worst-case gap search, no margin for a missed pass beyond the number you enter.
- **One payload, one mode.** A single rate and duty cycle. An instrument with a high-rate and a survey mode, or a second payload, has to be collapsed into one equivalent rate by hand.
- **Rates are achieved throughput.** Nothing here models modulation, coding, link margin, rain fade or protocol overhead. `800 Mbit/s` is an assertion about what the link delivers, and belongs to a link budget quicksat does not have.
- **No latency.** The budget answers whether the volume clears, never how old the oldest unsent frame is — which is often the requirement that actually binds.
- **Compression is a single ratio.** Constant across the scene, independent of content, with no lossy/lossless distinction.
- **Circular orbit.** Period from altitude, no eccentricity and no drag decay over the mission.